# Full-Fidelity Resolved-System Review Notebook

Interactive diagnostic notebook for `full_fidelity_binary_iterative_review.yaml`. Run cells one at a time to inspect the resolved truth/data and inference/reference systems before launching any larger campaign.

This notebook does not run a production campaign or optimization.


## 0. Setup and configuration

Set toggles here. The path resolver supports launching from the repository root or from `examples/notebooks`.


In [1]:
import os
import sys
import tempfile
from pathlib import Path

import jax
jax.config.update("jax_enable_x64", True)


def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in (p, *p.parents):
        if (q / "pyproject.toml").exists() or (q / ".git").exists():
            return q
    return p

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "dluxshera-matplotlib"))
REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

CONFIG_PATH = REPO_ROOT / "examples/recipes/full_fidelity_algorithm_campaign_template/full_fidelity_binary_iterative_review.yaml"
OUTPUT_ROOT = REPO_ROOT / "Results/full_fidelity_resolved_system_review"
RUN_LABEL = "interactive_review"
WRITE_ARTIFACTS = True
FAST_RENDER = True
ENABLE_NOISE_DEMOS = True

print("repo root:", REPO_ROOT)
print("PYTHONPATH has src:", str(SRC) in sys.path)
print("config path:", CONFIG_PATH)
print("output root:", OUTPUT_ROOT / RUN_LABEL)


repo root: /Users/dmckeith/Documents/GitHub/dluxshera-sandbox
PYTHONPATH has src: True
config path: /Users/dmckeith/Documents/GitHub/dluxshera-sandbox/examples/recipes/full_fidelity_algorithm_campaign_template/full_fidelity_binary_iterative_review.yaml
output root: /Users/dmckeith/Documents/GitHub/dluxshera-sandbox/Results/full_fidelity_resolved_system_review/interactive_review


### Plotting backend policy

For interactive pan/zoom plots in Jupyter Lab, install `ipympl` and set `PLOT_BACKEND = "widget"`. If `ipympl` is unavailable, use `PLOT_BACKEND = "inline"` for reliable static plots. Use `PLOT_BACKEND = "agg"` only for headless artifact generation.


In [ ]:
# Plotting backend policy
#
# Use "widget" for interactive pan/zoom in Jupyter Lab when ipympl is installed.
# Use "inline" for robust static plots in any notebook.
# Use "auto" to prefer widget and fall back to inline.
# Use "agg" only for headless artifact generation.
PLOT_BACKEND = "inline"  # options: "auto", "widget", "inline", "notebook", "agg"

import importlib.util


def configure_notebook_matplotlib_backend(mode="auto"):
    """Configure Matplotlib for interactive notebook review."""
    try:
        ip = get_ipython()
    except NameError:
        ip = None

    if mode == "agg":
        import matplotlib

        matplotlib.use("Agg", force=True)
        return "agg"

    if ip is None:
        # Not running in IPython/Jupyter. Leave backend alone.
        return "unchanged_non_ipython"

    if mode in {"auto", "widget"}:
        if importlib.util.find_spec("ipympl") is not None:
            ip.run_line_magic("matplotlib", "widget")
            return "widget"
        if mode == "widget":
            print("ipympl is not installed; falling back to inline backend.")

    if mode == "notebook":
        ip.run_line_magic("matplotlib", "notebook")
        return "notebook"

    ip.run_line_magic("matplotlib", "inline")
    return "inline"


ACTIVE_MATPLOTLIB_BACKEND = configure_notebook_matplotlib_backend(PLOT_BACKEND)
print(f"Matplotlib notebook backend: {ACTIVE_MATPLOTLIB_BACKEND}")


In [2]:
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.ion()

from dluxshera.utils import full_fidelity_review as review

cfg = review.load_smoke_config(CONFIG_PATH)
exp = cfg["experiment"]
outdir = OUTPUT_ROOT / RUN_LABEL
outdir.mkdir(parents=True, exist_ok=True)

print("schema_version:", exp.get("schema_version"))
print("run_name:", exp.get("run_name"))
print("source_kind:", exp.get("source_kind"))
print("target:", exp.get("target"))
print("system_preset:", exp.get("system_preset"))
print("output directory:", outdir)


schema_version: full_fidelity_binary_iterative_review.v1
run_name: full_fidelity_binary_iterative_review
source_kind: binary_target
target: ALPHA_CEN
system_preset: SHERA_FLIGHT_3P
output directory: /Users/dmckeith/Documents/GitHub/dluxshera-sandbox/Results/full_fidelity_resolved_system_review/interactive_review


In [ ]:
import matplotlib

print("Matplotlib backend:", matplotlib.get_backend())
print("Interactive mode:", plt.isinteractive())


## 0a. Config contract registry

Display registry-backed enum/option validation for the active config.


In [3]:
from dluxshera.utils.full_fidelity_config_schema import registry_entry_for_path, iter_string_fields, validate_config_contract

contract = validate_config_contract(cfg, config_tier='review', strict=False)
print('contract findings:', len(contract['findings']))
for finding in contract['findings']:
    print(f"{finding['severity']} {finding['field_path']} {finding['code']}: {finding['message']}")

rows = []
for path, value in iter_string_fields(cfg):
    pattern, entry = registry_entry_for_path(path)
    rows.append({
        'field_path': path,
        'value': value,
        'registry_pattern': pattern,
        'valid_values': ', '.join((entry or {}).get('valid_values', {}).keys()),
        'implemented_status': (entry or {}).get('implemented_status'),
    })
display(pd.DataFrame(rows))


contract findings: 0


,field_path,value,registry_pattern,valid_values,implemented_status
0,experiment.kind,full_fidelity_binary_iterative_review,experiment.kind,"full_fidelity_binary_iterative_review, full_fi...",implemented
1,experiment.schema_version,full_fidelity_binary_iterative_review.v1,experiment.schema_version,,implemented
2,experiment.run_name,full_fidelity_binary_iterative_review,experiment.run_name,,implemented
3,experiment.source_kind,binary_target,experiment.source_kind,"binary_target, binary, alpha_cen",implemented
4,experiment.target,ALPHA_CEN,experiment.target,,implemented
...,...,...,...,...,...
67,experiment.prior_draws.sigmas.optics.plate_sca...,fractional,experiment.prior_draws.sigmas.*.kind,"absolute, fractional, ppm, percent",implemented
68,experiment.prior_draws.sigmas.optics.primary.z...,absolute,experiment.prior_draws.sigmas.*.kind,"absolute, fractional, ppm, percent",implemented
69,experiment.prior_draws.sigmas.optics.primary.z...,nm,experiment.prior_draws.sigmas.*.unit,,implemented
70,experiment.prior_draws.sigmas.optics.secondary...,absolute,experiment.prior_draws.sigmas.*.kind,"absolute, fractional, ppm, percent",implemented


## 1. Translate smoke config and build model split

This uses the smoke wrapper's private translator by file-path import, then builds the package `CampaignModelSplit` without launching subblock inference.


In [4]:
ctx = review.build_model_split_from_smoke(cfg, outdir, run_label=RUN_LABEL, write_artifacts=WRITE_ARTIFACTS)
translated = ctx["translated_config"]
base_system = ctx["base_system_cfg"]
truth_system = ctx["truth_system_cfg"]
inference_system = ctx["inference_system_cfg"]
split = ctx["model_split"]

print("translated kind:", translated["experiment"].get("kind"))
print("truth hash:", split.truth_config_hash)
print("inference hash:", split.inference_config_hash)
print("matched:", split.truth_config_hash == split.inference_config_hash)
print("model split components:")
print(json.dumps(split.enabled_components, indent=2))


translated kind: observation_bias_campaign
truth hash: 34001eb056b453f93fcf1a7f94ef669bea0f1fcd21f2e22cd529ba0d9064f1a4
inference hash: ec954213b6df583f1765f35c1301b396456fcc17803b5f20f5d24c12464be816
matched: False
model split components:
{
  "spectral_model": {
    "enabled": true,
    "truth_label": null,
    "inference_label": null,
    "matched": false,
    "artifact_root": "/Users/dmckeith/Documents/GitHub/dluxshera-sandbox/Results/full_fidelity_resolved_system_review/interactive_review/model_split/spectral"
  },
  "high_order_wfe": {
    "enabled": true,
    "truth_label": "high_order_truth",
    "inference_label": "knowledge_error",
    "matched": false,
    "artifact_root": "/Users/dmckeith/Documents/GitHub/dluxshera-sandbox/Results/full_fidelity_resolved_system_review/interactive_review/model_split/high_order_wfe",
    "warnings": []
  },
  "scalar_reference_offsets": {
    "enabled": false,
    "n_offsets": 0
  },
  "trajectory_smear": {
    "enabled": true,
    "mode": "met

In [5]:
for role, system in [("base", base_system), ("truth", truth_system), ("inference", inference_system)]:
    s = review.summarize_source_config(system)
    print(f"{role} source wavelength_m / bandwidth_m / n_lambda:", s["wavelength_m"], s["bandwidth_m"], s["n_lambda"])
    print("  explicit wavelengths_m / weights / component_weights:", s["has_wavelengths_m"], s["has_weights"], s["has_component_weights"])

print("\nOriginal review config excerpt:")
print(json.dumps({k: exp.get(k) for k in ["spectral_model", "high_order_wfe", "subblocks", "iterative"]}, indent=2)[:5000])
print("\nTranslated observation-bias config excerpt:")
print(json.dumps(translated["experiment"], indent=2)[:5000])


base source wavelength_m / bandwidth_m / n_lambda: 5.5e-07 4.1e-08 3
  explicit wavelengths_m / weights / component_weights: False False False
truth source wavelength_m / bandwidth_m / n_lambda: 6e-07 5e-07 31
  explicit wavelengths_m / weights / component_weights: True False True
inference source wavelength_m / bandwidth_m / n_lambda: 6e-07 5e-07 7
  explicit wavelengths_m / weights / component_weights: True False True

Original review config excerpt:
{
  "spectral_model": {
    "enabled": true,
    "preserve_flux_parameters": true,
    "photometry_mode": "preserve_detected_flux_parameters",
    "source_seds": {
      "mode": "target",
      "generic_binary_fallback": "alpha_cen"
    },
    "truth": {
      "label": "truth_review_alpha_cen",
      "mode": "effective_source_spectrum",
      "n_lambda": 31,
      "wavelength_min_nm": 350.0,
      "wavelength_max_nm": 850.0,
      "components": {
        "detector_qe": {
          "enabled": true,
          "path": "data/detector_qe/LTN4

## 2. Spectral model review

Answers: actual truth/reference wavelength grids, `fast` clamp, inference wavelength override, component SED/weight differences, response-curve availability, and flux-factor provenance.


In [6]:
spectral_summary = review.summarize_spectral_deck(split)
spectral_tables = review.spectral_review_tables(base_system, truth_system, inference_system)
truth_spec = pd.DataFrame(spectral_tables["truth"])
inf_spec = pd.DataFrame(spectral_tables["inference"])
responses = review.response_curve_review(translated["experiment"].get("spectral_model"))

print(json.dumps({k: v for k, v in spectral_summary.items() if k != "provenance"}, indent=2, default=str))
print("truth rows:", len(truth_spec), "inference rows:", len(inf_spec))
print("detector QE enabled/available:", responses["detector_qe"]["enabled"], responses["detector_qe"]["available"])
print("M2 filter enabled/available:", responses["m2_filter_response"]["enabled"], responses["m2_filter_response"]["available"])

if WRITE_ARTIFACTS:
    review.write_spectral_review_csv(outdir / "spectral_review_tables.csv", spectral_tables)

display(truth_spec)
display(inf_spec)


{
  "truth": {
    "kind": "binary_target",
    "target": "ALPHA_CEN",
    "wavelength_m": 6e-07,
    "bandwidth_m": 5e-07,
    "n_lambda": 31,
    "has_wavelengths_m": true,
    "has_weights": false,
    "has_component_weights": true,
    "wavelengths_nm": [
      350.00000000000006,
      366.66666666666674,
      383.33333333333337,
      400.00000000000006,
      416.66666666666674,
      433.3333333333334,
      450.00000000000006,
      466.66666666666674,
      483.3333333333334,
      500.00000000000006,
      516.6666666666667,
      533.3333333333335,
      550.0,
      566.6666666666669,
      583.3333333333334,
      600.0000000000001,
      616.6666666666667,
      633.3333333333335,
      650.0,
      666.6666666666669,
      683.3333333333334,
      700.0000000000001,
      716.6666666666667,
      733.3333333333335,
      750.0,
      766.6666666666669,
      783.3333333333334,
      800.0000000000001,
      816.6666666666667,
      833.3333333333335,
      850.0
    ],

,role,component,index,wavelength_nm,weight
0,truth,primary,0,350.000000,0.000449
1,truth,primary,1,366.666667,0.000540
2,truth,primary,2,383.333333,0.000875
3,truth,primary,3,400.000000,0.001148
4,truth,primary,4,416.666667,0.001386
...,...,...,...,...,...
57,truth,secondary,26,783.333333,0.001736
58,truth,secondary,27,800.000000,0.001161
59,truth,secondary,28,816.666667,0.001400
60,truth,secondary,29,833.333333,0.001251


,role,component,index,wavelength_nm,weight
0,inference,primary,0,350.000000,0.008174
1,inference,primary,1,433.333333,0.029474
2,inference,primary,2,516.666667,0.869611
3,inference,primary,3,600.000000,0.011507
4,inference,primary,4,683.333333,0.044663
5,inference,primary,5,766.666667,0.021968
6,inference,primary,6,850.000000,0.014602
7,inference,secondary,0,350.000000,0.005822
8,inference,secondary,1,433.333333,0.025791
9,inference,secondary,2,516.666667,0.849055


In [7]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (label, response) in zip(axes[0], responses.items()):
    if response["available"]:
        ax.plot(response["wavelengths_nm"], response["response"])
    ax.set_title(f"{label}: {'active' if response['enabled'] else 'available but not active'}")
    ax.set_xlabel("wavelength [nm]")
    ax.set_ylabel("response")

for role, df, ax in [("truth", truth_spec, axes[1,0]), ("inference", inf_spec, axes[1,1])]:
    for comp, group in df.groupby("component"):
        ax.plot(group["wavelength_nm"], group["weight"], marker="o", label=comp)
    ax.set_title(f"Effective {role} component weights")
    ax.set_xlabel("wavelength [nm]")
    ax.set_ylabel("normalized sample weight")
    ax.legend()
plt.tight_layout()
plt.show()


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for comp in sorted(set(truth_spec["component"])):
    t = truth_spec[truth_spec["component"] == comp]
    i = inf_spec[inf_spec["component"] == comp]
    axes[0].plot(t["wavelength_nm"], t["weight"], marker="o", label=f"truth {comp}")
    axes[0].plot(i["wavelength_nm"], i["weight"], marker="s", linestyle="--", label=f"inference {comp}")

if {"primary", "secondary"} <= set(truth_spec["component"]):
    tp = truth_spec[truth_spec["component"] == "primary"].sort_values("wavelength_nm")
    ts = truth_spec[truth_spec["component"] == "secondary"].sort_values("wavelength_nm")
    axes[1].plot(tp["wavelength_nm"], tp["weight"].to_numpy() - ts["weight"].to_numpy(), marker="o", label="truth primary-secondary")
if {"primary", "secondary"} <= set(inf_spec["component"]):
    ip = inf_spec[inf_spec["component"] == "primary"].sort_values("wavelength_nm")
    isec = inf_spec[inf_spec["component"] == "secondary"].sort_values("wavelength_nm")
    axes[1].plot(ip["wavelength_nm"], ip["weight"].to_numpy() - isec["weight"].to_numpy(), marker="s", label="inference primary-secondary")
axes[0].set_title("Truth vs inference effective spectral response")
axes[0].set_xlabel("wavelength [nm]")
axes[0].set_ylabel("weight")
axes[0].legend()
axes[1].set_title("Component weight difference")
axes[1].set_xlabel("wavelength [nm]")
axes[1].set_ylabel("primary - secondary")
axes[1].legend()
plt.tight_layout()
plt.show()


In [9]:
def effective_rows(df):
    rows = []
    for comp, group in df.groupby("component"):
        w = group["weight"].to_numpy(float)
        lam = group["wavelength_nm"].to_numpy(float)
        rows.append({"component": comp, "sum_weights": w.sum(), "effective_wavelength_nm": (w * lam).sum() / w.sum()})
    return pd.DataFrame(rows)

print("Truth effective wavelengths")
display(effective_rows(truth_spec))
print("Inference effective wavelengths")
display(effective_rows(inf_spec))
print("Flux/provenance entries")
print(json.dumps(spectral_summary["provenance"], indent=2, default=str)[:8000])


Truth effective wavelengths


,component,sum_weights,effective_wavelength_nm
0,primary,1.0,550.685882
1,secondary,1.0,552.312541


Inference effective wavelengths


,component,sum_weights,effective_wavelength_nm
0,primary,1.0,531.610338
1,secondary,1.0,538.021629


Flux/provenance entries
{
  "schema_version": "spectral_source_config.v1",
  "truth": {
    "applied_to": "source",
    "role": "truth",
    "source_kind": "binary_target",
    "target": "ALPHA_CEN",
    "spectrum": {
      "source_kind": "binary_target",
      "component_labels": [
        "primary",
        "secondary"
      ],
      "n_lambda": 31,
      "component_spectra": {
        "primary": {
          "spectrum_label": "truth_primary",
          "source_kind": "binary_target",
          "component_labels": [
            "primary"
          ],
          "n_lambda": 31,
          "flux_factor": 4645588498.987997,
          "lambda_eff_nm": 550.685881957448,
          "bandwidth_rms_nm": 33.52079156482448,
          "weights_sum": 1.0,
          "weight_normalization": "sample_sum",
          "wavelength_unit": "m",
          "flux_factor_usage": "diagnostic_provenance_only",
          "log_flux_total_and_contrast": "preserved_detected_post_response_band_integrated",
          "s

## 3. Preserve-flux-parameters review

Verifies that spectral deck patching changes chromatic shape and provenance, not scalar band-integrated flux parameters, when `preserve_flux_parameters: true`.


In [10]:
flux_review = review.preserve_flux_review(base_system, truth_system, inference_system, translated["experiment"].get("spectral_model"))
print(json.dumps(flux_review, indent=2, default=str))
for warning in flux_review["warnings"]:
    print("WARNING:", warning)

for role, system in [("truth", truth_system), ("inference", inference_system)]:
    s = review.summarize_source_config(system)
    print(role, "component row sums:", s["component_weight_sums"])
    assert all(np.isclose(v, 1.0) for v in s["component_weight_sums"]), role


{
  "preserve_flux_parameters": true,
  "base": {
    "log_flux_total": null,
    "contrast": null
  },
  "truth": {
    "log_flux_total": null,
    "contrast": null
  },
  "inference": {
    "log_flux_total": null,
    "contrast": null
  },
  "warnings": [
    "preserve_flux_parameters is configured; consumption should be verified against spectral provenance."
  ]
}
truth component row sums: [1.0, 1.0]
inference component row sums: [1.0000000000000002, 1.0]


## 4. High-order WFE review

`npix=16` is smoke-only and should not be used for serious WFE studies. A practical next step is to match the generated high-order map sampling to the resolved pupil sampling (`optics.pupil_npix`) when feasible, or document a downsample/upsample policy.


In [11]:
wfe_summary = review.summarize_wfe_artifacts(split)
print("enabled:", wfe_summary.get("enabled"))
print("warnings:", wfe_summary.get("warnings"))
for mirror, item in wfe_summary.get("mirrors", {}).items():
    print("\n", mirror)
    print(" requested truth RMS:", item["requested_truth_rms_nm"], "measured:", item["truth_stats"]["rms_nm"])
    print(" requested error RMS:", item["requested_knowledge_error_rms_nm"], "measured:", item["knowledge_error_stats"]["rms_nm"])
    print(" low-order error coeffs:", item["zernike_coefficients_nm"]["error"])
    print(" index mapping:", item["noll_index_mapping"])
    if item["truth_opd_nm"].shape[0] < int(truth_system["optics"].get("pupil_npix", 0)):
        print("WARNING: WFE npix is much smaller than optics pupil_npix", item["truth_opd_nm"].shape, truth_system["optics"].get("pupil_npix"))


enabled: True
warnings: []

 primary
 requested truth RMS: 20.0 measured: 16.217802165127505
 requested error RMS: 0.3 measured: 0.3
 low-order error coeffs: {'Z4': -6.978195041283737e-17, 'Z5': 1.9621801206784422e-16, 'Z6': -3.582368887965913e-17, 'Z7': -1.9609194082304415e-17, 'Z8': -1.5816976334604677e-16, 'Z9': 2.6670207965638428e-17, 'Z10': -4.877148292255234e-17, 'Z11': 4.905797089396496e-17}
 index mapping: {'Z4': 0, 'Z5': 1, 'Z6': 2, 'Z7': 3, 'Z8': 4, 'Z9': 5, 'Z10': 6, 'Z11': 7}

 secondary
 requested truth RMS: 20.0 measured: 17.472623898345716
 requested error RMS: 0.3 measured: 0.29999999999999993
 low-order error coeffs: {'Z4': 2.6124199370440598e-17, 'Z5': -9.235953396346881e-17, 'Z6': 4.412527132713151e-19, 'Z7': -4.916988402444963e-17, 'Z8': 1.42339440643342e-16, 'Z9': 6.467768306753354e-17, 'Z10': 3.6577912138640995e-17, 'Z11': -4.6539803158966784e-17}
 index mapping: {'Z4': 0, 'Z5': 1, 'Z6': 2, 'Z7': 3, 'Z8': 4, 'Z9': 5, 'Z10': 6, 'Z11': 7}


In [25]:
for mirror, item in wfe_summary.get("mirrors", {}).items():
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    maps = [
        ("truth OPD [nm]", item["truth_opd_nm"]),
        ("inference/reference OPD [nm]", item["inference_opd_nm"]),
        ("knowledge error [nm]", item["knowledge_error_opd_nm"]),
        ("mask", item["mask"].astype(float)),
    ]
    for ax, (title, arr) in zip(axes.flat[:4], maps):
        im = ax.imshow(arr, origin="lower", cmap="RdBu_r" if "mask" not in title else "gray")
        ax.set_title(f"{mirror} {title}")
        plt.colorbar(im, ax=ax, shrink=0.8)
    vals = item["mask"]
    axes[1,1].hist(item["truth_opd_nm"][vals].ravel(), bins=25, alpha=0.5, label="truth")
    axes[1,1].hist(item["inference_opd_nm"][vals].ravel(), bins=25, alpha=0.5, label="inference")
    axes[1,1].hist(item["knowledge_error_opd_nm"][vals].ravel(), bins=25, alpha=0.5, label="error")
    axes[1,1].set_title("masked OPD histogram")
    axes[1,1].legend()
    coeff = item["zernike_coefficients_nm"]
    labels = list(coeff["truth"].keys())
    x = np.arange(len(labels))
    axes[1,2].bar(x - 0.25, [coeff["truth"][k] for k in labels], width=0.25, label="truth")
    axes[1,2].bar(x, [coeff["inference"][k] for k in labels], width=0.25, label="inference")
    axes[1,2].bar(x + 0.25, [coeff["error"][k] for k in labels], width=0.25, label="error")
    axes[1,2].set_xticks(x, labels, rotation=45)
    axes[1,2].set_title("low-order Zernike fits")
    axes[1,2].legend()
    plt.tight_layout()
    plt.show()


/var/folders/b5/w3q_2b855kjb0d6fwp5r7cmc0000gq/T/ipykernel_39574/4276870518.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/b5/w3q_2b855kjb0d6fwp5r7cmc0000gq/T/ipykernel_39574/4276870518.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Optics and system-preset review


In [13]:
optics_rows = pd.DataFrame(review.optics_diff_table(base_system, truth_system, inference_system))
print("Base optics")
print(json.dumps(review.summarize_optics_config(base_system), indent=2, default=str))
print("Truth optics")
print(json.dumps(review.summarize_optics_config(truth_system), indent=2, default=str))
print("Inference optics")
print(json.dumps(review.summarize_optics_config(inference_system), indent=2, default=str))
display(optics_rows)


Base optics
{
  "preset": "SHERA_FLIGHT_3P",
  "kind": "three_plane",
  "psf_npix": 256,
  "oversample": 3,
  "pupil_npix": 256,
  "plate_scale_as_per_pix": null,
  "primary_noll_indices": [
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11
  ],
  "secondary_noll_indices": [
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11
  ],
  "high_order_wfe_enabled": false,
  "diffractive_pupil_path": "src/dluxshera/data/pupils/diffractive_pupil.npy",
  "special_keys": [
    "dp_design_wavelength_m",
    "dp_path",
    "pupil_npix"
  ]
}
Truth optics
{
  "preset": "SHERA_FLIGHT_3P",
  "kind": "three_plane",
  "psf_npix": 256,
  "oversample": 3,
  "pupil_npix": 256,
  "plate_scale_as_per_pix": null,
  "primary_noll_indices": [
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11
  ],
  "secondary_noll_indices": [
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11
  ],
  "high_order_wfe_enabled": true,
  "diffractive_pupil_path": "src/dluxshera/data/pupils/diffractive_pupil.

,config_path,base_value,truth_value,inference_value,status
0,optics.kind,three_plane,three_plane,three_plane,matched
1,optics.psf_npix,256,256,256,matched
2,optics.oversample,3,3,3,matched
3,optics.pupil_npix,256,256,256,matched
4,optics.plate_scale_as_per_pix,None,None,None,matched
5,optics.primary_noll_indices,"[4, 5, 6, 7, 8, 9, 10, 11]","[4, 5, 6, 7, 8, 9, 10, 11]","[4, 5, 6, 7, 8, 9, 10, 11]",matched
6,optics.secondary_noll_indices,"[4, 5, 6, 7, 8, 9, 10, 11]","[4, 5, 6, 7, 8, 9, 10, 11]","[4, 5, 6, 7, 8, 9, 10, 11]",matched
7,optics.high_order_wfe.enabled,None,True,True,matched
8,optics.dp_path,src/dluxshera/data/pupils/diffractive_pupil.npy,src/dluxshera/data/pupils/diffractive_pupil.npy,src/dluxshera/data/pupils/diffractive_pupil.npy,matched


## 6. Detector layer and calibration-map review

Absent maps are reported clearly rather than treated as errors.


In [14]:
det_truth = review.summarize_detector_config(truth_system)
det_inf = review.summarize_detector_config(inference_system)
print("truth detector:")
print(json.dumps(det_truth, indent=2, default=str))
print("inference detector:")
print(json.dumps(det_inf, indent=2, default=str))
display(pd.DataFrame(det_truth["layers"]))
cal_maps = review.load_detector_calibration_maps(truth_system)
print("loaded calibration maps:", list(cal_maps))
if not cal_maps:
    print("No detector calibration maps loaded or implemented for this smoke path.")


truth detector:
{
  "model": "HWK4123",
  "pixel_pitch_m": 4.6e-06,
  "array_size": [
    4096,
    2300
  ],
  "read_noise": 0.5,
  "dark_current": 2.0,
  "layers": [
    {
      "index": 0,
      "name": "downsample",
      "kind": "Downsample",
      "key_parameters": {
        "kernel_size": 3
      }
    },
    {
      "index": 1,
      "name": "pixel_offsets",
      "kind": "ApplyPixelOffsets",
      "key_parameters": {
        "dx_path": "src/dluxshera/data/pixel_offsets/dx_baseline.fits",
        "dy_path": "src/dluxshera/data/pixel_offsets/dy_baseline.fits"
      }
    },
    {
      "index": 2,
      "name": "pixel_response",
      "kind": "ApplyPixelResponse",
      "key_parameters": {
        "prf_path": "src/dluxshera/data/pixel_response/prf_baseline.fits"
      }
    },
    {
      "index": 3,
      "name": "jitter",
      "kind": "ApplyJitter",
      "key_parameters": {
        "sigma": 1e-12,
        "kernel_size": 3
      }
    }
  ],
  "calibration_paths": [
    {
   

,index,name,kind,key_parameters
0,0,downsample,Downsample,{'kernel_size': 3}
1,1,pixel_offsets,ApplyPixelOffsets,{'dx_path': 'src/dluxshera/data/pixel_offsets/...
2,2,pixel_response,ApplyPixelResponse,{'prf_path': 'src/dluxshera/data/pixel_respons...
3,3,jitter,ApplyJitter,"{'sigma': 1e-12, 'kernel_size': 3}"


loaded calibration maps: ['pixel_offsets.dx_path', 'pixel_offsets.dy_path', 'pixel_response.prf_path']


In [15]:
if cal_maps:
    n = len(cal_maps)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), squeeze=False)
    for ax, (name, arr) in zip(axes.flat, cal_maps.items()):
        im = ax.imshow(arr, origin="lower")
        ax.set_title(name)
        plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()


## 7. Noise model review

The smoke uses `noise: disabled`. The demo below is controlled and does not modify the config file.


In [16]:
noise_summary = review.summarize_noise_config(translated, truth_system)
print(json.dumps(noise_summary, indent=2, default=str))
if ENABLE_NOISE_DEMOS:
    ndemo = review.noise_demo(seed=123, read_noise=float(noise_summary.get("read_noise") or 0.5))
    print(json.dumps(ndemo["diagnostics"], indent=2))


{
  "noise_mode": "enabled",
  "structured_request": {
    "enabled": true,
    "shot_noise": true,
    "read_noise": true,
    "dark_current": false,
    "use_detector_read_noise": true,
    "read_noise_electrons": null,
    "dark_current_e_per_s": null,
    "variance_floor": 1.0,
    "seed_policy": "from_subblock_noise_seed"
  },
  "shot_noise_enabled": true,
  "read_noise_enabled": true,
  "read_noise": 0.5,
  "read_noise_unit": "electrons RMS per pixel",
  "dark_current_enabled": false,
  "dark_current": 2.0,
  "dark_current_unit": "electrons / s / pixel",
  "variance_floor": 1.0,
  "use_render_variance": "inherited/default",
  "separate_term_control": false
}
{
  "shot_residual_var": 39.664893182317776,
  "read_residual_var": 0.261633407681288,
  "combined_residual_var": 40.07837608789015,
  "mean_model_variance": 40.1303819608641
}


In [17]:
if ENABLE_NOISE_DEMOS:
    fig, axes = plt.subplots(2, 4, figsize=(15, 7))
    for ax, key in zip(axes[0], ["noiseless", "shot", "read", "combined"]):
        im = ax.imshow(ndemo[key], origin="lower")
        ax.set_title(key)
        plt.colorbar(im, ax=ax, shrink=0.75)
    for ax, key in zip(axes[1, :3], ["shot", "read", "combined"]):
        ax.hist((ndemo[key] - ndemo["noiseless"]).ravel(), bins=40)
        ax.set_title(f"{key} residual")
    im = axes[1,3].imshow(ndemo["combined_variance"], origin="lower")
    axes[1,3].set_title("combined variance map")
    plt.colorbar(im, ax=axes[1,3], shrink=0.75)
    plt.tight_layout()
    plt.show()


## 8. Trajectory review and 15 s high-pass filter

High-pass filtering is currently diagnostic-only here. Production `trajectory_processing.high_pass_filter.enabled=true` is reserved/not implemented by trajectory-smear parsing.


In [18]:
trajectory_review = review.load_trajectory_for_review(translated)
hp_review = review.make_high_pass_trajectory_diagnostic(trajectory_review, timescale_s=15.0)
print(json.dumps(trajectory_review.get("summary", trajectory_review), indent=2, default=str))
print("high-pass note:", hp_review.get("note"))


{
  "raw_span_s": [
    0.0,
    1800.0
  ],
  "raw_sample_count": 18001,
  "selected_start_s": 60.0,
  "selected_end_s": 61.2,
  "n_subblocks": 2,
  "n_frames": 10,
  "output_keys": [
    "source.x_position_as",
    "source.y_position_as",
    "source.position_angle_deg"
  ]
}
high-pass note: Notebook diagnostic only; campaign trace source currently rejects high_pass_filter.enabled=true.


In [19]:
if trajectory_review.get("available"):
    traj = trajectory_review["trajectory"]
    fig, axes = plt.subplots(3, 2, figsize=(14, 10))
    for ax, key in zip(axes[:,0], traj.values):
        ax.plot(traj.time_s, traj.values[key], lw=1)
        ax.axvspan(trajectory_review["frame_times_s"][0], trajectory_review["frame_times_s"][-1], color="orange", alpha=0.25)
        ax.set_title(f"raw full-window {key}")
    for ax, key in zip(axes[:,1], traj.values):
        for block in trajectory_review["blocks"]:
            ax.plot(block.frame_times_s, block.truth[key], marker="o", label="truth" if block.subblock_index == 0 else None)
            ax.plot(block.frame_times_s, block.prediction[key], linestyle="--", label="linear fit" if block.subblock_index == 0 else None)
        ax.set_title(f"selected segment and per-subblock fit {key}")
        ax.legend()
    plt.tight_layout()
    plt.show()


In [20]:
if trajectory_review.get("available"):
    fig, axes = plt.subplots(3, 2, figsize=(14, 10))
    for ax, key in zip(axes[:,0], trajectory_review["trajectory"].values):
        for block in trajectory_review["blocks"]:
            ax.plot(block.frame_times_s, block.residual[key], marker="o")
        ax.set_title(f"per-subblock residual {key}")
    for ax, key in zip(axes[:,1], hp_review["series"]):
        series = hp_review["series"][key]
        ax.plot(trajectory_review["trajectory"].time_s, series["raw"], label="raw", alpha=0.6)
        ax.plot(trajectory_review["trajectory"].time_s, series["low_pass"], label="15 s low-pass")
        ax.plot(trajectory_review["trajectory"].time_s, series["high_pass"], label="high-pass")
        ax.set_title(f"{key}; high-pass RMS={series['rms_high_pass']:.4g}")
        ax.legend()
    plt.tight_layout()
    plt.show()


## 9. Trace jitter review

Explicit question: does `trace_jitter` add noise on top of the trajectory?


In [21]:
trace_jitter_review = review.compare_trace_jitter_enabled_disabled(translated)
print(json.dumps(trace_jitter_review, indent=2, default=str))
display(pd.DataFrame([{"key": k, "rms_difference": v} for k, v in trace_jitter_review.get("rms_difference", {}).items()]))


{
  "status": "downstream_template_override",
  "is_additive_to_materialized_trajectory_csv": false,
  "is_ignored": false,
  "jitter_config": {
    "enabled": true,
    "apply_to": "downstream_template_override",
    "x_sigma_as": 0.001,
    "y_sigma_as": 0.001,
    "pa_sigma_deg": 0.0001
  },
  "rms_difference": {
    "source.x_position_as": 0.0,
    "source.y_position_as": 0.0,
    "source.position_angle_deg": 0.0
  },
  "conclusion": "Trajectory frame_truth.csv is generated without trace_jitter. The observation-bias wrapper forwards trace_jitter as run_obs_subblock_study CLI overrides, where it only changes an existing iid_jitter/random_walk effect in the trace template. In trajectory mode with external frame truth, this is downstream behavior and not visible in the materialized trajectory CSV."
}


,key,rms_difference
0,source.x_position_as,0.0
1,source.y_position_as,0.0
2,source.position_angle_deg,0.0


## 10. Model rendering sanity checks

Optional hook. This notebook keeps rendering lightweight and does not run a campaign.


In [22]:
render_review = review.render_tiny_review_images(truth_system, inference_system, fast=FAST_RENDER)
print(json.dumps(render_review, indent=2))
if not render_review.get("available"):
    print("Rendering skipped:", render_review.get("reason"))


{
  "available": false,
  "reason": "Tiny rendering is intentionally optional; no production campaign is launched by this helper."
}
Rendering skipped: Tiny rendering is intentionally optional; no production campaign is launched by this helper.


## 11. Summary dashboard


In [23]:
dashboard = pd.DataFrame(review.summary_dashboard(
    config=translated,
    base_cfg=base_system,
    truth_cfg=truth_system,
    inference_cfg=inference_system,
    model_split=split,
    trajectory_review=trajectory_review,
    trace_jitter_review=trace_jitter_review,
))
display(dashboard)


,Component,Status,Truth setting,Inference setting,Difference / mismatch,Reviewer action
0,source target / component SEDs,review,ALPHA_CEN,ALPHA_CEN,{'truth_primary_minus_secondary_lambda_eff_nm'...,inspect SED/weights
1,spectral grid,mismatch,31,7,fast clamp/reference band,confirm acceptable
2,QE,configured,"{'enabled': True, 'path': 'data/detector_qe/LT...",{'mode': 'same_as_truth'},see response review,inspect
3,M2 filter,configured,"{'enabled': True, 'path': 'data/filter_respons...",{'mode': 'same_as_truth'},see response review,inspect
4,flux parameters,preserved,"{'log_flux_total': None, 'contrast': None}","{'log_flux_total': None, 'contrast': None}",scalar band-integrated parameters,verify
5,high-order WFE maps,enabled,truth maps,truth + knowledge error,"{'enabled': True, 'truth_label': 'high_order_t...",inspect RMS/maps
6,low-order Zernike coefficients,review,"[4, 5, 6, 7, 8, 9, 10, 11]","[4, 5, 6, 7, 8, 9, 10, 11]",active index mapping starts at array index 0,verify mapping
7,optics preset,review,"{'preset': 'SHERA_FLIGHT_3P', 'kind': 'three_p...","{'preset': 'SHERA_FLIGHT_3P', 'kind': 'three_p...",see diff table,decide if new preset needed
8,detector layers,matched,"[{'index': 0, 'name': 'downsample', 'kind': 'D...","[{'index': 0, 'name': 'downsample', 'kind': 'D...",see detector maps,inspect
9,calibration maps,present,"[{'layer': 'pixel_offsets', 'kind': 'ApplyPixe...","[{'layer': 'pixel_offsets', 'kind': 'ApplyPixe...",matched config paths,inspect maps if present


In [24]:
if WRITE_ARTIFACTS:
    artifact_paths = review.write_review_artifacts(
        outdir,
        base_system_cfg=base_system,
        truth_system_cfg=truth_system,
        inference_system_cfg=inference_system,
        model_split=split,
        spectral_summary=spectral_summary,
        wfe_summary=wfe_summary,
        detector_summary=det_truth,
        noise_summary=noise_summary,
        trajectory_summary=trajectory_review.get("summary", trajectory_review),
    )
    print(json.dumps(artifact_paths, indent=2))


{
  "resolved_base_system.yaml": "/Users/dmckeith/Documents/GitHub/dluxshera-sandbox/Results/full_fidelity_resolved_system_review/interactive_review/resolved_base_system.yaml",
  "resolved_truth_system.yaml": "/Users/dmckeith/Documents/GitHub/dluxshera-sandbox/Results/full_fidelity_resolved_system_review/interactive_review/resolved_truth_system.yaml",
  "resolved_inference_system.yaml": "/Users/dmckeith/Documents/GitHub/dluxshera-sandbox/Results/full_fidelity_resolved_system_review/interactive_review/resolved_inference_system.yaml",
  "model_split_summary.json": "/Users/dmckeith/Documents/GitHub/dluxshera-sandbox/Results/full_fidelity_resolved_system_review/interactive_review/model_split_summary.json",
  "spectral_review_summary.json": "/Users/dmckeith/Documents/GitHub/dluxshera-sandbox/Results/full_fidelity_resolved_system_review/interactive_review/spectral_review_summary.json",
  "wfe_review_summary.json": "/Users/dmckeith/Documents/GitHub/dluxshera-sandbox/Results/full_fidelity_reso

## Reviewer notes / decisions

- Source / spectral deck decision:
- WFE map decision:
- Detector / calibration decision:
- Noise decision:
- Trajectory / high-pass decision:
- Trace jitter decision:
- Config changes to make before campaign:
- Follow-up tasks:
